# 1. Bài toán

Hãy tưởng tượng một tình huống rất đời thường, có một bình luận: "Mấy thằng ngu này nói chuyện chán thật", nếu làm phân loại văn bản thông thường, máy chỉ có thể trả lời bình luận này có nội dung xúc phạm. 

Nhưng người kiểm duyệt vẫn phải tự đọc cả câu để xem xúc phạm ở chỗ nào. Bài toán ViHOS muốn máy làm thêm một bước: "Mấy thằng [ngu] này nói chuyện chán thật". Máy phải chỉ chính xác đoạn nào có vấn đề, đó là ý tưởng cốt lõi của toàn bộ đồ án. Nguồn ViHOS chính thức cũng mô tả mục tiêu là phát hiện trực tiếp các hate/offensive spans trong bình luận, thay vì chỉ gắn một nhãn cho toàn bộ câu

# 2. Input / Output

Input: Một bình luận tiếng Việt, ví dụ: "Ca sĩ này hát như dở hơi"

Output: Những đoạn từ mang tính thù ghét hoặc xúc phạm, ví dụ: "Ca sĩ này hát như [dở hơi]". Hay có câu không gì xấu: "Bài hát này nghe cũng ổn" thì output không có đoạn nào cần đánh dấu. 

# 3. Span

Span = một đoạn trong câu, có thể chỉ là một từ: "Mày đúng là [ngu]", hoặc nhiều từ: "Mày đúng là [đồ ngu ngốc]". Phần nằm trong [...] chính là span mà mô hình cần tìm. Vì vậy: Hate/Offensive Span Detection = tìm các đoạn mang nội dung thù ghét/xúc phạm". ViHOS có hơn 11 nghìn bình luận và các span này được con người đánh dấu sẵn

# 4. Điểm quan trọng nhất: ta không chỉ phân loại cả câu

| Bài toán | Câu | Kết quả |
|---|---|---|
| Phân loại bình luận | `Mày đúng là đồ ngu` | `OFFENSIVE` |
| **Bài toán của ta** | `Mày đúng là đồ ngu` | `đồ ngu` |

Đây là khác biệt quan trọng nhất, ví dụ câu dài hơn: "Tôi đồng ý với phần đầu, nhưng thằng đó đúng là đồ vô dụng, phần còn lại thì khá ổn", nếu chỉ phân loại: OFFENSIVE thì ta không biết vì sao. Còn bài toán của ta: "Tôi đồng ý với phần đầu, nhưng thằng đó đúng là [đồ vô dụng], phần còn lại thì khá ổn => rõ ràng hơn rất nhiều

Đó cũng chính là động cơ mà nhóm tác giả ViHOS đưa ra: việc highlight đoạn có vấn đề giúp người kiểm duyệt hiểu lý do tốt hơn so với chỉ nhận một nhãn cho toàn comment

# 5. Thuộc loại bài toán NLP nào ?

Tên dễ hiểu: Phát hiện đoạn văn bản (Span Detection), và cách chúng ta dự kiến giải nó thuộc nhóm: Sequence Labeling - gán nhãn chuỗi. Gán nhãn chuỗi thực chật chỉ là đi qua từng từ và nói xem từ đó có thuộc đoạn cần tìm hay không

Ví dụ: Mày đúng là đồ ngu

| Từ | Máy cần hiểu |
|---|---|
| Mày | bình thường |
| đúng | bình thường |
| là | bình thường |
| đồ | bắt đầu đoạn xúc phạm |
| ngu | nằm tiếp trong đoạn xúc phạm |

Sau này ta sẽ viết thành:

| Từ | Nhãn |
|---|---|
| Mày | O |
| đúng | O |
| là | O |
| đồ | B |
| ngu | I |

Chỉ cần hiểu: Máy nhìn từng phần của câu để tìm xem đoạn xấu bắt đầu ở đâu và kết thúc ở đâu

# 6. Tại sao bài toán này không đơn giản là như tìm từ chửi

Giả sử ta làm một danh sách: ngu, dốt, đần... rồi thấy từ nào xuất hiện thì đánh dấu, nghe có vẻ dễ. Nhưng thực tế người dùng mạng xã hội có thể viết: "nguuu", "n.g.u", "ngu vcl", "não cá vàng", hoặc dùng từ nói bóng gió, chơi chữ. Thậm chí cùng một từ có thể có vấn đề trong ngữ cảnh này nhưng lại không có vấn đề trong ngữ cảnh khác. ViHOS được tạo ra không chỉ như một từ điển từ xấu; tác giả nhấn mạnh họ muốn đánh dấu ý nghĩa hoặc nội dung hate/offensive trong ngữ cảnh, và dataset có cả hiện tượng teencode, ẩn dụ, hoán dụ, chơi chữ... Đây sẽ là phần rất hay để chúng ta phân tích sau này

# 7. Toàn bộ hệ thống sau này có thể hình dung như sau

Bình luận tiếng Việt => Mày đúng là đồ ngu thật => Mô hình NLP xem từng phần câu => "Mày đúng là [đồ ngu] thật" => Kết quả